# Week 3 Objectives

1. Baseline computation — a single number per signal per ride (avg HR and avg power from minutes 5–15)
1. Delta features — current value minus baseline (how far has HR drifted from its anchor?)
1. Rolling statistics — mean and slope over a sliding time window (is the drift accelerating?)

### Task 1 — Compute the per-ride baseline
From your clean records_filled DataFrame, extract all rows where the elapsed time since the first timestamp is between 5 and 15 minutes. Compute:

- baseline_hr — mean heart rate in that window
- baseline_power — mean power in that window

Acceptance criteria:

- Both are single float values, not a DataFrame
- You can print them and they look physiologically plausible (HR somewhere 110–150, power somewhere 100–250 for a typical ride)
- You handle the case where the baseline window contains is_pause = True rows — those should be excluded

### Task 2 — Compute delta features
Add two new columns to your DataFrame:

hr_delta = heart_rate minus baseline_hr
power_delta = power minus baseline_power

Acceptance criteria:

Both columns exist across all 6708 rows
Values near the baseline window should hover close to 0
Values late in the ride should show meaningful deviation

### Task 3 - Rolling mean and slope
Add rolling statistics over a 5-minute window (= 300 rows, since you have 1 record/second):

hr_rolling_mean — rolling average of heart_rate over 300 seconds
hr_rolling_slope — this one is harder: the trend direction of HR over the window

For the slope, here's a hint: you're looking for whether HR is going up or down over time. One way to think about it — if you plotted HR over the last 5 minutes and drew a line of best fit, what would the gradient of that line be? Look up rolling().apply() in pandas and numpy.polyfit.
Acceptance criteria:

Both columns exist
First ~300 rows will have NaN in both — that's expected and correct, explain why in a comment
hr_rolling_slope is positive when HR is rising and negative when falling — verify this manually on a short slice of data

First of all I'm going to define a function to import a a fit file. And get the records.

In [1]:
from fitparse import FitFile
import pandas as pd
import numpy as np

def get_records (fit_file,record_type,amount=None):
    all_records = [] ## Defines an array for all messages in the file
    for count,message in enumerate(fit_file.get_messages (record_type)): ## iterates through all messages of a certain type
        if amount is not None and count >= amount: ## sets a limit in case the user wants a limit
            break
        ## here I define what I want to see, a message is structured in fields. Therfore I want to have an dictionary with all fields of a message
        field_content = {
            field.name : field.value
            for field in message if field.name and 'unknown' not in field.name
        }
        all_records.append (field_content)
    return all_records

def get_dataframe (fit_file,record_type,index=None):
    if index:
        df = pd.DataFrame(get_records(fit_file,record_type)).set_index(index)
    else:
        df = pd.DataFrame(get_records(fit_file,record_type))
    return df

def find_gaps (timestamps):
    time_gap = []
    for count,timestamp in enumerate(timestamps):
        if count > 0:
            timestampdiff = timestamp - timestamps[count-1] 
            if timestampdiff.total_seconds() > 1:
                time_gap.append ({
                    'start_time': timestamps[count-1],
                    'end_time': timestamp,
                    'gap': timestampdiff.total_seconds()
                })
    return time_gap

def get_cleaned_records (fit_file,check_gaps=False):
    records = get_dataframe (fit_file,'record','timestamp')
    if check_gaps:
        print (find_gaps(records.index))
    records['cadence'] = records['cadence'].ffill(limit=10)
    records['heart_rate'] = records['heart_rate'].ffill(limit=3)
    records['power'] = records['power'].ffill(limit=1)
    return records

def get_baseline_timestamps(timestamps,baseline_minute_start,baseline_minute_end):
    if baseline_minute_start < 0 or baseline_minute_end <=baseline_minute_start:
        raise ValueError ('Invalid baseline windows negative values are not allowed')    
    ride_start = timestamps[0]

    baseline_start = ride_start + pd.Timedelta(minutes=baseline_minute_start)
    baseline_end = ride_start + pd.Timedelta(minutes=baseline_minute_end)
    return [baseline_start,baseline_end]
        
def get_baselines (parameter,dataFrame,baseline_start,baseline_end):
    baseline_timestamps = get_baseline_timestamps(sorted(dataFrame.index),baseline_start,baseline_end)
    baselines = []
    for p in parameter:
        baselines.append({
            'name': p,
            'value':dataFrame[p][baseline_timestamps[0]:baseline_timestamps[1]].mean()})
    return baselines

def add_delta_features (dataFrame,baselines):
    df = dataFrame.copy()
    for b in baselines:
        df[f'delta_{b['name']}'] = df[f'{b['name']}'] - b['value'] 
    return df

def get_slope (window):
    return np.polyfit(np.arange(len(window)),window,1)[0]

def add_slope_and_mean (dataFrame,parameters,rolling_length):
# The first 'rolling_length' rows will be NaN as the rolling function is configured to need at least
# the number of data points which is equal to the paramter length.
    df = dataFrame.copy()
    for p in parameters:
        df[f'{p}_slope'] = df[f'{p}'].rolling(rolling_length).apply(get_slope)
        df[f'{p}_mean'] = df[f'{p}'].rolling(rolling_length).mean()
    return df


activity = FitFile('22105359350_ACTIVITY.fit')

records = get_cleaned_records(activity)

baselines = get_baselines (['heart_rate','power'],records,5,15)

records = add_delta_features (records,baselines)

records = add_slope_and_mean (records,['heart_rate','power'],300)

In [2]:
window = records['heart_rate_slope'].nlargest(10).index[5]

records_index = records.index.get_loc(window)

print (records[['heart_rate','heart_rate_slope']].loc[records.index[records_index-300]:records.index[records_index]])




                     heart_rate  heart_rate_slope
timestamp                                        
2026-03-08 15:08:59       139.0         -0.157688
2026-03-08 15:09:00       140.0         -0.156690
2026-03-08 15:09:01       139.0         -0.155683
2026-03-08 15:09:02       138.0         -0.154732
2026-03-08 15:09:03       137.0         -0.153770
...                         ...               ...
2026-03-08 15:14:44       161.0          0.135392
2026-03-08 15:14:45       160.0          0.135455
2026-03-08 15:14:46       158.0          0.135442
2026-03-08 15:14:47       156.0          0.135287
2026-03-08 15:14:48       155.0          0.135059

[301 rows x 2 columns]


### Task 4 HR/Power Ratio Delta

One new column:

hr_power_ratio_delta = (heart_rate / power) - (baseline_hr / baseline_power)

You have everything you need already in the script — the baselines list and the DataFrame with all features.
Before you code, one question:
Some power values in your data are 0 — coasting, stopped at a light. What happens mathematically when you divide by 0, and what should the ratio delta value be at those moments?
Think about what a zero-power moment means physiologically for drift detection, then decide how to handle it and write a comment in the code explaining your reasoning. That decision is part of the acceptance criteria.

In [41]:
## hr_power_ratio = heart_rate / power

def add_hr_power_delta_ratio (dataFrame,baseline):
    df = dataFrame.copy()

    hr_power_ratio_delta = (df['heart_rate']) / df['power'] - (baseline[0]['value'] / baseline[1]['value'])

    df ['hr_power_ratio_delta'] = hr_power_ratio_delta.replace([np.inf, -np.inf],np.nan)

    return df

records_with_ratio = add_hr_power_delta_ratio(records,baselines)

print (records_with_ratio[['heart_rate','power','hr_power_ratio_delta']])

        

                     heart_rate  power  hr_power_ratio_delta
timestamp                                                   
2026-03-08 13:24:47       108.0   11.0              8.873506
2026-03-08 13:24:48       108.0    0.0                   NaN
2026-03-08 13:24:49       108.0    0.0                   NaN
2026-03-08 13:24:50       108.0    0.0                   NaN
2026-03-08 13:24:51       108.0    0.0                   NaN
...                         ...    ...                   ...
2026-03-08 15:25:17       111.0    0.0                   NaN
2026-03-08 15:25:18       110.0    0.0                   NaN
2026-03-08 15:25:19       109.0    0.0                   NaN
2026-03-08 15:25:20       109.0    0.0                   NaN
2026-03-08 15:25:21       108.0    0.0                   NaN

[6708 rows x 3 columns]
